# Pseudo-Top State Revision

Fold-safe second-stage correction of pseudo-top elevation errors before regime probabilities are formed.

## 1. Purpose And Model

The plateau audit shows persistent hard wells across regime-mean variants, suggesting the bottleneck is pseudo-top/state error rather than blend tuning. This notebook adds a conservative second-stage top residual corrector: after fold-safe pseudo-top predictions are made, train residual models on top prediction errors from the training folds and apply shrinked corrections before recomputing formation-regime probabilities.

## 2. Setup

In [1]:
"Configure imports, paths, and deterministic settings."

from pathlib import Path
import json
import shutil
import time

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import HuberRegressor, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor

pd.set_option("display.max_columns", 180)
pd.set_option("display.width", 220)

ROOT = Path.cwd()
if not (ROOT / "context").exists():
    ROOT = next(parent for parent in ROOT.parents if (parent / "context").exists())

RUN_DATE = "2026-07-01"
RUN_TAG = "pseudo_top_state_revision"
RUN_ID = f"{RUN_DATE}__{RUN_TAG}"
SEED = 20260630
FORMATION_COLS = ["ANCC", "ASTNU", "ASTNL", "EGFDU", "EGFDL", "BUDA"]
PRIMARY_FOLD = "strict_hybrid_fold"
STRESS_FOLD = "analog_distance_fold"
MAX_TARGET_ROWS_PER_WELL = 120
ENSEMBLE_SEEDS = [20260630, 20260631]

DATA_DIR = ROOT / "data" / "raw"
TRAIN_DIR = DATA_DIR / "train"
TABLE_DIR = ROOT / "artifacts" / "tables"
REPORT_DIR = ROOT / "artifacts" / "reports"
RUN_DIR = ROOT / "artifacts" / "logs" / "runs" / RUN_ID
for directory in [TABLE_DIR, REPORT_DIR, RUN_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "run_id": RUN_ID,
    "seed": SEED,
    "formation_cols": FORMATION_COLS,
    "primary_fold": PRIMARY_FOLD,
    "stress_fold": STRESS_FOLD,
    "max_target_rows_per_well": MAX_TARGET_ROWS_PER_WELL,
    "ensemble_seeds": ENSEMBLE_SEEDS,
    "current_public_best_rmse": 14.472,
    "best_xgb_known_state_local_rmse": 13.4334,
    "best_xgb_known_state_stress_rmse": 13.7533,
    "promotion_margin": 0.05,
    "baseline_regime_mean_rmse": 13.092783707491265,
    "baseline_regime_mean_stress_rmse": 13.400574608144202,
    "plateau_audit": "2026-07-01__plateau_failure_slice_audit",
}
with (RUN_DIR / "config.json").open("w", encoding="utf-8") as handle:
    json.dump(CONFIG, handle, indent=2)

print(ROOT)
print(RUN_DIR)

C:\Users\sardo\OneDrive\Desktop\Classes\Projects\Wellbore-tvt-Modeling
C:\Users\sardo\OneDrive\Desktop\Classes\Projects\Wellbore-tvt-Modeling\artifacts\logs\runs\2026-07-01__pseudo_top_state_revision


## 3. Data And Fold-Safe Pseudo Tops

In [2]:
"Build sampled target-zone rows with legal predictors and direct top-elevation labels."

folds = pd.read_csv(TABLE_DIR / "validation__strict_cv_fold_assignments__2026-06-26.csv")
descriptors = pd.read_csv(TABLE_DIR / "model__typewell_alignment_feature_descriptors__2026-06-23.csv")
well_meta = folds.merge(descriptors, on="wellname", how="left")
well_meta = well_meta.drop_duplicates("wellname").reset_index(drop=True)

DESCRIPTOR_COLS = [
    "known_rows", "known_tvt_span", "known_md_span", "known_gr_mean_last300", "known_gr_std_last300", "known_gr_slope_last300",
    "xy_span_known", "z_drop_known", "typewell_corr_last300", "typewell_mae_last300", "typewell_best_lag_corr",
    "typewell_best_lag_tvt", "typewell_best_lag_mae", "selected_analog_distance", "extrapolation_stress_score",
    "analog_stress_score", "combined_stress_score", "descriptor_cluster",
]


def safe_divide(num, den):
    return num / (den + 1e-9)


def load_typewell(directory, wellname):
    path = Path(directory) / f"{wellname}__typewell.csv"
    if not path.exists():
        return pd.DataFrame(columns=["TVT", "GR", "Geology"])
    return pd.read_csv(path)


def interp_typewell_gr(typewell, tvt_values):
    if typewell.empty or not {"TVT", "GR"}.issubset(typewell.columns):
        return np.full(len(tvt_values), np.nan)
    clean = typewell[["TVT", "GR"]].dropna().sort_values("TVT")
    if len(clean) < 5:
        return np.full(len(tvt_values), np.nan)
    return np.interp(np.asarray(tvt_values, dtype=float), clean["TVT"].to_numpy(), clean["GR"].to_numpy(), left=np.nan, right=np.nan)


def build_target_rows(csv_path, meta_row):
    df = pd.read_csv(csv_path).sort_values("MD").reset_index(drop=True)
    known_mask = df["TVT_input"].notna()
    if known_mask.sum() == 0 or (~known_mask).sum() == 0:
        return pd.DataFrame()
    last_known_idx = int(np.flatnonzero(known_mask.to_numpy())[-1])
    known = df.loc[:last_known_idx].copy()
    target = df.loc[last_known_idx + 1:].copy()
    target = target.loc[target["TVT"].notna()].copy()
    if target.empty:
        return pd.DataFrame()
    if len(target) > MAX_TARGET_ROWS_PER_WELL:
        sample_idx = np.linspace(0, len(target) - 1, MAX_TARGET_ROWS_PER_WELL).round().astype(int)
        target = target.iloc[np.unique(sample_idx)].copy()

    typewell = load_typewell(csv_path.parent, meta_row["wellname"])
    last = known.iloc[-1]
    first = df.iloc[0]
    known_gr = known["GR"].astype(float).clip(0, 300)
    known_gr_finite = known_gr.dropna()
    gr_fill = float(known_gr.tail(300).median()) if known_gr.notna().any() else 0.0
    target_gr = target["GR"].astype(float).clip(0, 300).fillna(gr_fill)

    known_md_span_state = float(known["MD"].iloc[-1] - known["MD"].iloc[0]) if len(known) > 1 else 0.0
    known_tvt_delta_state = float(known["TVT_input"].iloc[-1] - known["TVT_input"].iloc[0]) if len(known) > 1 else 0.0
    known_tvt_slope_state = known_tvt_delta_state / max(abs(known_md_span_state), 1.0)
    known_xy_span_state = float(np.hypot(known["X"].iloc[-1] - known["X"].iloc[0], known["Y"].iloc[-1] - known["Y"].iloc[0])) if len(known) > 1 else 0.0
    known_z_drop_state = float(known["Z"].iloc[-1] - known["Z"].iloc[0]) if len(known) > 1 else 0.0
    known_gr_coverage_state = float(known_gr.notna().mean()) if len(known_gr) else 0.0
    known_gr_last_state = float(known_gr_finite.iloc[-1]) if len(known_gr_finite) else gr_fill
    known_gr_iqr_state = float(known_gr_finite.quantile(0.75) - known_gr_finite.quantile(0.25)) if len(known_gr_finite) >= 4 else 0.0

    target = target.reset_index(drop=False).rename(columns={"index": "row_idx_original"})
    out = pd.DataFrame({
        "wellname": meta_row["wellname"],
        "row_idx_original": target["row_idx_original"].astype(float),
        "MD": target["MD"].astype(float),
        "X": target["X"].astype(float),
        "Y": target["Y"].astype(float),
        "Z": target["Z"].astype(float),
        "GR": target_gr.to_numpy(),
        "TVT": target["TVT"].astype(float),
        "last_known_tvt_feature": float(last["TVT"]),
        "last_known_md": float(last["MD"]),
        "last_known_x": float(last["X"]),
        "last_known_y": float(last["Y"]),
        "last_known_z": float(last["Z"]),
        "first_md": float(first["MD"]),
        "state_known_tvt_last": float(last["TVT_input"] if pd.notna(last.get("TVT_input", np.nan)) else last["TVT"]),
        "state_known_tvt_slope": known_tvt_slope_state,
        "state_known_gr_coverage": known_gr_coverage_state,
        "state_known_gr_last": known_gr_last_state,
        "state_known_gr_iqr": known_gr_iqr_state,
        "state_known_xy_span": known_xy_span_state,
        "state_known_z_drop": known_z_drop_state,
        "state_known_md_span": known_md_span_state,
    })
    out["target_delta"] = out["TVT"] - out["last_known_tvt_feature"]
    out["md_from_last_known"] = out["MD"] - out["last_known_md"]
    out["md_from_start"] = out["MD"] - out["first_md"]
    out["eval_progress"] = safe_divide(np.arange(1, len(out) + 1), len(out))
    out["eval_progress_sq"] = out["eval_progress"] ** 2
    out["dx_from_last"] = out["X"] - out["last_known_x"]
    out["dy_from_last"] = out["Y"] - out["last_known_y"]
    out["dz_from_last"] = out["Z"] - out["last_known_z"]
    out["horizontal_from_last"] = np.sqrt(out["dx_from_last"] ** 2 + out["dy_from_last"] ** 2)
    out["distance_from_last"] = np.sqrt(out["horizontal_from_last"] ** 2 + out["dz_from_last"] ** 2)
    out["distance_log1p"] = np.log1p(out["distance_from_last"].clip(lower=0))
    out["tvt_linear_proxy"] = out["last_known_tvt_feature"] + out["md_from_last_known"]
    best_lag = float(meta_row.get("typewell_best_lag_tvt", 0.0) if pd.notna(meta_row.get("typewell_best_lag_tvt", np.nan)) else 0.0)
    out["typewell_gr_linear_tvt"] = interp_typewell_gr(typewell, out["tvt_linear_proxy"].to_numpy())
    out["typewell_gr_lag_best"] = interp_typewell_gr(typewell, out["tvt_linear_proxy"].to_numpy() + best_lag)
    out["gr_minus_typewell_linear"] = out["GR"] - out["typewell_gr_linear_tvt"]
    out["gr_minus_typewell_bestlag"] = out["GR"] - out["typewell_gr_lag_best"]

    for formation in FORMATION_COLS:
        out[f"formation_top_{formation}"] = target[formation].astype(float).to_numpy()
        out[f"label_signed_offset_{formation}"] = out["Z"] - out[f"formation_top_{formation}"]
        out[f"label_abs_offset_{formation}"] = np.abs(out[f"label_signed_offset_{formation}"])

    abs_label_cols = [f"label_abs_offset_{formation}" for formation in FORMATION_COLS]
    out["label_nearest_abs_offset"] = out[abs_label_cols].min(axis=1)
    out["label_nearest_formation"] = out[abs_label_cols].idxmin(axis=1).str.replace("label_abs_offset_", "", regex=False)
    out["label_mean_abs_offset"] = out[abs_label_cols].mean(axis=1)

    for col in [PRIMARY_FOLD, STRESS_FOLD, "descriptor_cluster_fold", "extrapolation_fold"]:
        out[col] = int(meta_row[col])
    for col in DESCRIPTOR_COLS:
        out[col] = float(meta_row[col]) if pd.notna(meta_row[col]) else np.nan
    return out


frames = []
for meta_row in well_meta.to_dict(orient="records"):
    csv_path = TRAIN_DIR / f"{meta_row['wellname']}__horizontal_well.csv"
    if csv_path.exists():
        frame = build_target_rows(csv_path, meta_row)
        if not frame.empty:
            frames.append(frame)

data = pd.concat(frames, ignore_index=True)

SAFE_ROW_COLS = [
    "row_idx_original", "MD", "X", "Y", "Z", "GR", "last_known_tvt_feature", "last_known_md",
    "md_from_last_known", "md_from_start", "eval_progress", "eval_progress_sq",
    "dx_from_last", "dy_from_last", "dz_from_last", "horizontal_from_last", "distance_from_last", "distance_log1p",
]
SAFE_FEATURE_COLS = [col for col in [*SAFE_ROW_COLS, *DESCRIPTOR_COLS] if col in data.columns]
TYPEWELL_FEATURE_COLS = [
    "tvt_linear_proxy", "typewell_gr_linear_tvt", "typewell_gr_lag_best", "gr_minus_typewell_linear", "gr_minus_typewell_bestlag",
]
TYPEWELL_FEATURE_COLS = [col for col in TYPEWELL_FEATURE_COLS if col in data.columns]
STATE_FEATURE_COLS = [
    "state_known_tvt_last", "state_known_tvt_slope", "state_known_gr_coverage", "state_known_gr_last",
    "state_known_gr_iqr", "state_known_xy_span", "state_known_z_drop", "state_known_md_span",
]
DIRECT_FORMATION_FEATURE_COLS = [
    *[f"label_signed_offset_{formation}" for formation in FORMATION_COLS],
    *[f"label_abs_offset_{formation}" for formation in FORMATION_COLS],
    "label_nearest_abs_offset", "label_mean_abs_offset",
]

top_label_summary = []
for formation in FORMATION_COLS:
    values = data[f"formation_top_{formation}"].astype(float)
    offsets = data[f"label_signed_offset_{formation}"].astype(float)
    top_label_summary.append({
        "formation": formation,
        "top_mean": float(values.mean()),
        "top_std": float(values.std()),
        "top_min": float(values.min()),
        "top_max": float(values.max()),
        "signed_offset_mean": float(offsets.mean()),
        "signed_offset_std": float(offsets.std()),
        "abs_offset_mean": float(np.abs(offsets).mean()),
    })
top_label_summary = pd.DataFrame(top_label_summary)
label_counts = data["label_nearest_formation"].value_counts().rename_axis("formation").reset_index(name="nearest_label_rows")
top_label_summary = top_label_summary.merge(label_counts, on="formation", how="left").fillna({"nearest_label_rows": 0})

label_path = TABLE_DIR / "model__pseudo_top_state_revision_top_label_summary__2026-07-01.csv"
top_label_summary.to_csv(label_path, index=False)
shutil.copy2(label_path, RUN_DIR / label_path.name)

feature_preview_path = TABLE_DIR / "model__pseudo_top_state_revision_feature_preview__2026-07-01.csv"
data.head(500).to_csv(feature_preview_path, index=False)
shutil.copy2(feature_preview_path, RUN_DIR / feature_preview_path.name)

print(data.shape)
print("safe_features", len(SAFE_FEATURE_COLS))
print(top_label_summary.to_string(index=False))

(92760, 81)
safe_features 36
formation     top_mean    top_std   top_min  top_max  signed_offset_mean  signed_offset_std  abs_offset_mean  nearest_label_rows
     ANCC -9008.099049 633.914859 -10647.31 -7301.76         -416.172460          55.411650       416.172460                 0.0
    ASTNU -9190.780447 639.418693 -10852.79 -7509.20         -244.653750          63.560087       244.653750                 0.0
    ASTNL -9255.299324 624.857096 -10865.58 -7614.43         -180.134874          51.475069       180.135674              2455.0
    EGFDU -9349.765034 619.827567 -10887.58 -7704.03          -85.669164          43.889718        91.053770              3919.0
    EGFDL -9391.937722 619.776843 -10932.35 -7731.61          -43.798726          44.572447        55.375403             50858.0
     BUDA -9516.381964 633.862561 -11104.73 -7835.57           80.947766          45.807212        80.953400             35528.0


## 4. Pseudo-Top Probabilities

In [3]:
"Train fold-safe top-elevation ensembles and derive uncertainty, margin, and nearest-top features."

PSEUDO_TOP_PREDICTOR_COLS = [
    *SAFE_FEATURE_COLS,
    *TYPEWELL_FEATURE_COLS,
    *STATE_FEATURE_COLS,
]
PSEUDO_TOP_PREDICTOR_COLS = list(dict.fromkeys([col for col in PSEUDO_TOP_PREDICTOR_COLS if col in data.columns]))

pseudo_params = {
    "objective": "reg:squarederror",
    "n_estimators": 220,
    "learning_rate": 0.040,
    "max_depth": 3,
    "min_child_weight": 90,
    "subsample": 0.85,
    "colsample_bytree": 0.85,
    "reg_alpha": 0.20,
    "reg_lambda": 10.0,
    "tree_method": "hist",
    "n_jobs": 4,
    "verbosity": 0,
}


def rmse(y_true, y_pred):
    return float(mean_squared_error(y_true, y_pred) ** 0.5)


def softmax_neg_abs(abs_matrix, scale=50.0):
    logits = -abs_matrix / scale
    logits = logits - np.nanmax(logits, axis=1, keepdims=True)
    exp_logits = np.exp(logits)
    return exp_logits / np.clip(exp_logits.sum(axis=1, keepdims=True), 1e-12, None)


def add_pseudo_features_for_fold_column(frame, fold_col):
    out = frame.copy()
    proxy_quality_rows = []
    for formation in FORMATION_COLS:
        out[f"pseudo_top_elev_{formation}__{fold_col}"] = np.nan
        out[f"pseudo_top_std_{formation}__{fold_col}"] = np.nan

    X_all = out[PSEUDO_TOP_PREDICTOR_COLS].replace([np.inf, -np.inf], np.nan)
    for fold in sorted(out[fold_col].dropna().unique()):
        valid_mask = out[fold_col].eq(fold)
        train_mask = ~valid_mask
        X_train = X_all.loc[train_mask]
        X_valid = X_all.loc[valid_mask]
        for formation in FORMATION_COLS:
            target_col = f"formation_top_{formation}"
            label_train_mask = train_mask & out[target_col].notna()
            label_valid_mask = valid_mask & out[target_col].notna()
            seed_predictions = []
            if label_train_mask.sum() >= 20:
                X_label_train = X_all.loc[label_train_mask]
                y_train = out.loc[label_train_mask, target_col].astype(float)
                for seed in ENSEMBLE_SEEDS:
                    model = XGBRegressor(**pseudo_params, random_state=seed)
                    model.fit(X_label_train, y_train, verbose=False)
                    seed_predictions.append(model.predict(X_valid))
                pred_matrix = np.column_stack(seed_predictions)
                pred_mean = pred_matrix.mean(axis=1)
                pred_std = pred_matrix.std(axis=1)
            else:
                pred_mean = np.full(valid_mask.sum(), float(out[target_col].median()))
                pred_std = np.zeros(valid_mask.sum())
            out.loc[valid_mask, f"pseudo_top_elev_{formation}__{fold_col}"] = pred_mean
            out.loc[valid_mask, f"pseudo_top_std_{formation}__{fold_col}"] = pred_std
            if label_valid_mask.sum() > 0:
                valid_index = out.loc[valid_mask].index
                pred_series = pd.Series(pred_mean, index=valid_index).loc[label_valid_mask]
                y_valid = out.loc[label_valid_mask, target_col].astype(float)
                true_offset = out.loc[label_valid_mask, "Z"].to_numpy(float) - y_valid.to_numpy(float)
                pred_offset = out.loc[label_valid_mask, "Z"].to_numpy(float) - pred_series.to_numpy(float)
                corr = float(np.corrcoef(y_valid.to_numpy(float), pred_series.to_numpy(float))[0, 1]) if len(y_valid) > 2 else np.nan
                top_rmse = rmse(y_valid, pred_series)
                top_mae = float(mean_absolute_error(y_valid, pred_series))
                signed_offset_rmse = rmse(true_offset, pred_offset)
            else:
                corr = np.nan
                top_rmse = np.nan
                top_mae = np.nan
                signed_offset_rmse = np.nan
            proxy_quality_rows.append({
                "fold_column": fold_col,
                "fold": int(fold),
                "formation": formation,
                "n_train": int(label_train_mask.sum()),
                "n_valid": int(label_valid_mask.sum()),
                "top_rmse": top_rmse,
                "top_mae": top_mae,
                "top_corr": corr,
                "signed_offset_rmse": signed_offset_rmse,
                "mean_ensemble_std": float(np.mean(pred_std)),
            })

    for formation in FORMATION_COLS:
        top_col = f"pseudo_top_elev_{formation}__{fold_col}"
        std_col = f"pseudo_top_std_{formation}__{fold_col}"
        out[top_col] = out[top_col].fillna(float(out[f"formation_top_{formation}"].median()))
        out[std_col] = out[std_col].fillna(float(out[std_col].median()) if out[std_col].notna().any() else 0.0)
        out[f"pseudo_signed_offset_{formation}__{fold_col}"] = out["Z"] - out[top_col]
        out[f"pseudo_abs_offset_{formation}__{fold_col}"] = np.abs(out[f"pseudo_signed_offset_{formation}__{fold_col}"])

    abs_cols = [f"pseudo_abs_offset_{formation}__{fold_col}" for formation in FORMATION_COLS]
    std_cols = [f"pseudo_top_std_{formation}__{fold_col}" for formation in FORMATION_COLS]
    abs_matrix = out[abs_cols].to_numpy(float)
    probs = softmax_neg_abs(abs_matrix)
    sorted_abs = np.sort(abs_matrix, axis=1)
    sorted_probs = np.sort(probs, axis=1)[:, ::-1]
    nearest_idx = np.argmin(abs_matrix, axis=1)

    out[f"pseudo_nearest_abs_offset__{fold_col}"] = sorted_abs[:, 0]
    out[f"pseudo_second_abs_offset__{fold_col}"] = sorted_abs[:, 1]
    out[f"pseudo_top_margin__{fold_col}"] = out[f"pseudo_second_abs_offset__{fold_col}"] - out[f"pseudo_nearest_abs_offset__{fold_col}"]
    out[f"pseudo_top_confidence__{fold_col}"] = 1.0 / (1.0 + np.exp(-out[f"pseudo_top_margin__{fold_col}"].clip(-100, 100) / 25.0))
    out[f"pseudo_near_boundary_25__{fold_col}"] = (out[f"pseudo_nearest_abs_offset__{fold_col}"] <= 25.0).astype(float)
    out[f"pseudo_uncertain_boundary__{fold_col}"] = ((out[f"pseudo_nearest_abs_offset__{fold_col}"] <= 50.0) & (out[f"pseudo_top_margin__{fold_col}"] <= 25.0)).astype(float)
    out[f"pseudo_mean_abs_offset__{fold_col}"] = abs_matrix.mean(axis=1)
    out[f"pseudo_min_std__{fold_col}"] = out[std_cols].min(axis=1)
    out[f"pseudo_mean_std__{fold_col}"] = out[std_cols].mean(axis=1)
    out[f"pseudo_top1_prob__{fold_col}"] = sorted_probs[:, 0]
    out[f"pseudo_top2_prob__{fold_col}"] = sorted_probs[:, 1]
    out[f"pseudo_top_entropy__{fold_col}"] = -(probs * np.log(probs + 1e-12)).sum(axis=1)
    out[f"pseudo_nearest_formation__{fold_col}"] = [FORMATION_COLS[int(idx)] for idx in nearest_idx]
    out[f"pseudo_nearest_matches_label__{fold_col}"] = (out[f"pseudo_nearest_formation__{fold_col}"] == out["label_nearest_formation"]).astype(float)

    return out, pd.DataFrame(proxy_quality_rows)


pseudo_feature_cols_by_fold = {}
pseudo_uncertainty_cols_by_fold = {}
quality_parts = []
for fold_col in [PRIMARY_FOLD, STRESS_FOLD]:
    data, quality = add_pseudo_features_for_fold_column(data, fold_col)
    quality_parts.append(quality)
    pseudo_feature_cols_by_fold[fold_col] = [
        *[f"pseudo_signed_offset_{formation}__{fold_col}" for formation in FORMATION_COLS],
        *[f"pseudo_abs_offset_{formation}__{fold_col}" for formation in FORMATION_COLS],
        f"pseudo_nearest_abs_offset__{fold_col}",
        f"pseudo_second_abs_offset__{fold_col}",
        f"pseudo_mean_abs_offset__{fold_col}",
    ]
    pseudo_uncertainty_cols_by_fold[fold_col] = [
        *[f"pseudo_top_std_{formation}__{fold_col}" for formation in FORMATION_COLS],
        f"pseudo_top_margin__{fold_col}",
        f"pseudo_top_confidence__{fold_col}",
        f"pseudo_near_boundary_25__{fold_col}",
        f"pseudo_uncertain_boundary__{fold_col}",
        f"pseudo_min_std__{fold_col}",
        f"pseudo_mean_std__{fold_col}",
        f"pseudo_top1_prob__{fold_col}",
        f"pseudo_top2_prob__{fold_col}",
        f"pseudo_top_entropy__{fold_col}",
    ]

proxy_quality = pd.concat(quality_parts, ignore_index=True)
nearest_quality = []
for fold_col in [PRIMARY_FOLD, STRESS_FOLD]:
    nearest_quality.append({
        "fold_column": fold_col,
        "fold": "all",
        "formation": "nearest_classifier",
        "n_train": np.nan,
        "n_valid": int(len(data)),
        "top_rmse": np.nan,
        "top_mae": np.nan,
        "top_corr": np.nan,
        "signed_offset_rmse": np.nan,
        "mean_ensemble_std": float(data[f"pseudo_mean_std__{fold_col}"].mean()),
        "nearest_accuracy": float(data[f"pseudo_nearest_matches_label__{fold_col}"].mean()),
    })
proxy_quality["nearest_accuracy"] = np.nan
proxy_quality = pd.concat([proxy_quality, pd.DataFrame(nearest_quality)], ignore_index=True)

proxy_quality_path = TABLE_DIR / "model__pseudo_top_state_revision_proxy_quality__2026-07-01.csv"
proxy_quality.to_csv(proxy_quality_path, index=False)
shutil.copy2(proxy_quality_path, RUN_DIR / proxy_quality_path.name)
proxy_quality.groupby(["fold_column", "formation"], dropna=False).agg(top_rmse=("top_rmse", "mean"), top_mae=("top_mae", "mean"), mean_ensemble_std=("mean_ensemble_std", "mean"), nearest_accuracy=("nearest_accuracy", "mean")).reset_index()

,fold_column,formation,top_rmse,top_mae,mean_ensemble_std,nearest_accuracy
0,analog_distance_fold,ANCC,36.448121,27.561076,2.225590,NaN
1,analog_distance_fold,ASTNL,36.512561,27.158327,2.604606,NaN
2,analog_distance_fold,ASTNU,38.155576,28.416605,2.408923,NaN
3,analog_distance_fold,BUDA,34.737212,25.140603,2.262772,NaN
4,analog_distance_fold,EGFDL,34.746888,25.533790,2.353502,NaN
5,analog_distance_fold,EGFDU,34.494617,25.287956,2.264115,NaN
6,analog_distance_fold,nearest_classifier,NaN,NaN,2.353398,0.579743
7,strict_hybrid_fold,ANCC,35.638111,26.984800,1.995119,NaN
8,strict_hybrid_fold,ASTNL,35.637506,26.891537,2.281886,NaN
9,strict_hybrid_fold,ASTNU,37.183606,27.877467,2.611257,NaN


## 4b. Fold-Safe Pseudo-Top State Revision

In [4]:

"Correct fold-safe pseudo-top elevation predictions with conservative residual models."

REVISION_FEATURE_COLS = list(dict.fromkeys([
    *SAFE_FEATURE_COLS,
    *TYPEWELL_FEATURE_COLS,
    *STATE_FEATURE_COLS,
    "selected_analog_distance", "extrapolation_stress_score", "analog_stress_score", "combined_stress_score",
    "known_rows", "known_tvt_span", "known_md_span", "known_gr_mean_last300", "known_gr_std_last300",
    "typewell_corr_last300", "typewell_mae_last300", "typewell_best_lag_corr", "typewell_best_lag_mae",
]))
REVISION_FEATURE_COLS = [col for col in REVISION_FEATURE_COLS if col in data.columns]
revision_shrinkages = [0.25, 0.50, 0.75]


def fit_top_residual_model(X_train, y_train, X_valid):
    model = XGBRegressor(
        objective="reg:pseudohubererror", n_estimators=260, learning_rate=0.035, max_depth=2,
        min_child_weight=80, subsample=0.88, colsample_bytree=0.88, reg_alpha=0.30,
        reg_lambda=12.0, gamma=0.10, tree_method="hist", random_state=SEED, n_jobs=4, verbosity=0,
    )
    model.fit(X_train, y_train, verbose=False)
    return model.predict(X_valid)


def revise_pseudo_tops_for_fold(frame, fold_col):
    out = frame.copy()
    X_all = out[REVISION_FEATURE_COLS].replace([np.inf, -np.inf], np.nan)
    revision_rows = []
    for shrink in revision_shrinkages:
        for formation in FORMATION_COLS:
            out[f"revised_top_elev_{formation}_s{int(shrink*100):02d}__{fold_col}"] = out[f"pseudo_top_elev_{formation}__{fold_col}"]
            out[f"revised_abs_offset_{formation}_s{int(shrink*100):02d}__{fold_col}"] = out[f"pseudo_abs_offset_{formation}__{fold_col}"]
    for fold in sorted(out[fold_col].dropna().unique()):
        valid_mask = out[fold_col].eq(fold)
        train_mask = ~valid_mask
        for formation in FORMATION_COLS:
            pred_col = f"pseudo_top_elev_{formation}__{fold_col}"
            target_col = f"formation_top_{formation}"
            label_mask = train_mask & out[pred_col].notna() & out[target_col].notna()
            valid_label_mask = valid_mask & out[pred_col].notna() & out[target_col].notna()
            if label_mask.sum() < 100:
                correction = np.zeros(valid_mask.sum())
            else:
                y_resid = (out.loc[label_mask, target_col] - out.loc[label_mask, pred_col]).astype(float)
                correction = fit_top_residual_model(X_all.loc[label_mask], y_resid, X_all.loc[valid_mask])
            base_pred = out.loc[valid_mask, pred_col].astype(float).to_numpy()
            actual_valid = out.loc[valid_mask, target_col].astype(float).to_numpy()
            base_rmse = rmse(actual_valid[~np.isnan(actual_valid)], base_pred[~np.isnan(actual_valid)]) if valid_label_mask.sum() else np.nan
            for shrink in revision_shrinkages:
                revised = base_pred + shrink * correction
                revised_col = f"revised_top_elev_{formation}_s{int(shrink*100):02d}__{fold_col}"
                abs_col = f"revised_abs_offset_{formation}_s{int(shrink*100):02d}__{fold_col}"
                out.loc[valid_mask, revised_col] = revised
                out.loc[valid_mask, abs_col] = np.abs(out.loc[valid_mask, "Z"].astype(float).to_numpy() - revised)
                revised_rmse = rmse(actual_valid[~np.isnan(actual_valid)], revised[~np.isnan(actual_valid)]) if valid_label_mask.sum() else np.nan
                revision_rows.append({
                    "fold_column": fold_col,
                    "fold": int(fold),
                    "formation": formation,
                    "shrink": shrink,
                    "n_train": int(label_mask.sum()),
                    "n_valid": int(valid_label_mask.sum()),
                    "base_top_rmse": base_rmse,
                    "revised_top_rmse": revised_rmse,
                    "top_rmse_delta": revised_rmse - base_rmse if np.isfinite(base_rmse) and np.isfinite(revised_rmse) else np.nan,
                    "mean_abs_correction": float(np.nanmean(np.abs(correction))) if len(correction) else 0.0,
                })
    return out, pd.DataFrame(revision_rows)


revision_quality_parts = []
for active_fold in [PRIMARY_FOLD, STRESS_FOLD]:
    data, revision_quality = revise_pseudo_tops_for_fold(data, active_fold)
    revision_quality_parts.append(revision_quality)
revision_quality = pd.concat(revision_quality_parts, ignore_index=True)
print(revision_quality.groupby(["fold_column", "shrink"]).agg(top_rmse_delta=("top_rmse_delta", "mean"), mean_abs_correction=("mean_abs_correction", "mean")).reset_index().to_string(index=False))


         fold_column  shrink  top_rmse_delta  mean_abs_correction
analog_distance_fold    0.25        0.102221             8.074793
analog_distance_fold    0.50        0.399624             8.074793
analog_distance_fold    0.75        0.887413             8.074793
  strict_hybrid_fold    0.25       -0.006411             6.909991
  strict_hybrid_fold    0.50        0.132876             6.909991
  strict_hybrid_fold    0.75        0.415163             6.909991


## 5. Regime Probability Basis

In [5]:
"Create soft formation-regime probabilities and MAP residual basis features."


def add_probability_and_basis_features(frame, fold_col, scale=50.0, revision_shrink=None):
    out = frame
    if revision_shrink is None:
        abs_cols = [f"pseudo_abs_offset_{formation}__{fold_col}" for formation in FORMATION_COLS]
        suffix = "base"
    else:
        suffix = f"s{int(revision_shrink * 100):02d}"
        abs_cols = [f"revised_abs_offset_{formation}_{suffix}__{fold_col}" for formation in FORMATION_COLS]
    abs_matrix = out[abs_cols].to_numpy(float)
    logits = -abs_matrix / scale
    logits = logits - np.nanmax(logits, axis=1, keepdims=True)
    probs = np.exp(logits)
    probs = probs / np.clip(probs.sum(axis=1, keepdims=True), 1e-12, None)
    for idx, formation in enumerate(FORMATION_COLS):
        prob_col = f"mix_prob_{formation}__{fold_col}"
        out[prob_col] = probs[:, idx]
        out[f"mix_prob_{formation}_x_progress__{fold_col}"] = out[prob_col] * out["eval_progress"]
        out[f"mix_prob_{formation}_x_distance__{fold_col}"] = out[prob_col] * out["distance_log1p"]
        out[f"mix_prob_{formation}_x_near_top__{fold_col}"] = out[prob_col] * (out[f"pseudo_abs_offset_{formation}__{fold_col}"] <= 40.0).astype(float)
    out[f"mix_entropy__{fold_col}"] = -(probs * np.log(probs + 1e-12)).sum(axis=1)
    out[f"mix_top1_prob__{fold_col}"] = np.sort(probs, axis=1)[:, -1]
    out[f"mix_top2_prob__{fold_col}"] = np.sort(probs, axis=1)[:, -2]
    out[f"mix_prob_margin__{fold_col}"] = out[f"mix_top1_prob__{fold_col}"] - out[f"mix_top2_prob__{fold_col}"]
    out[f"mix_confidence__{fold_col}"] = 1.0 / (1.0 + np.exp((out[f"mix_entropy__{fold_col}"] - 1.15) / 0.12))
    return out


mix_basis_cols_by_fold = {}
mix_uncertainty_cols_by_fold = {}
revision_candidates = [None, 0.25, 0.50, 0.75]
for active_fold in [PRIMARY_FOLD, STRESS_FOLD]:
    for shrink in revision_candidates:
        data = add_probability_and_basis_features(data, active_fold, revision_shrink=shrink)
    mix_basis_cols_by_fold[active_fold] = []
    for formation in FORMATION_COLS:
        mix_basis_cols_by_fold[active_fold].extend([
            f"mix_prob_{formation}__{active_fold}",
            f"mix_prob_{formation}_x_progress__{active_fold}",
            f"mix_prob_{formation}_x_distance__{active_fold}",
            f"mix_prob_{formation}_x_near_top__{active_fold}",
        ])
    mix_uncertainty_cols_by_fold[active_fold] = [
        f"mix_entropy__{active_fold}",
        f"mix_top1_prob__{active_fold}",
        f"mix_top2_prob__{active_fold}",
        f"mix_prob_margin__{active_fold}",
        f"mix_confidence__{active_fold}",
        f"pseudo_top_margin__{active_fold}",
        f"pseudo_mean_std__{active_fold}",
        f"pseudo_top_entropy__{active_fold}",
    ]

print(len(mix_basis_cols_by_fold[PRIMARY_FOLD]), len(mix_uncertainty_cols_by_fold[PRIMARY_FOLD]))
data[mix_basis_cols_by_fold[PRIMARY_FOLD] + mix_uncertainty_cols_by_fold[PRIMARY_FOLD]].head()

24 8


,mix_prob_ANCC__strict_hybrid_fold,mix_prob_ANCC_x_progress__strict_hybrid_fold,mix_prob_ANCC_x_distance__strict_hybrid_fold,mix_prob_ANCC_x_near_top__strict_hybrid_fold,mix_prob_ASTNU__strict_hybrid_fold,mix_prob_ASTNU_x_progress__strict_hybrid_fold,mix_prob_ASTNU_x_distance__strict_hybrid_fold,mix_prob_ASTNU_x_near_top__strict_hybrid_fold,mix_prob_ASTNL__strict_hybrid_fold,mix_prob_ASTNL_x_progress__strict_hybrid_fold,mix_prob_ASTNL_x_distance__strict_hybrid_fold,mix_prob_ASTNL_x_near_top__strict_hybrid_fold,mix_prob_EGFDU__strict_hybrid_fold,mix_prob_EGFDU_x_progress__strict_hybrid_fold,mix_prob_EGFDU_x_distance__strict_hybrid_fold,mix_prob_EGFDU_x_near_top__strict_hybrid_fold,mix_prob_EGFDL__strict_hybrid_fold,mix_prob_EGFDL_x_progress__strict_hybrid_fold,mix_prob_EGFDL_x_distance__strict_hybrid_fold,mix_prob_EGFDL_x_near_top__strict_hybrid_fold,mix_prob_BUDA__strict_hybrid_fold,mix_prob_BUDA_x_progress__strict_hybrid_fold,mix_prob_BUDA_x_distance__strict_hybrid_fold,mix_prob_BUDA_x_near_top__strict_hybrid_fold,mix_entropy__strict_hybrid_fold,mix_top1_prob__strict_hybrid_fold,mix_top2_prob__strict_hybrid_fold,mix_prob_margin__strict_hybrid_fold,mix_confidence__strict_hybrid_fold,pseudo_top_margin__strict_hybrid_fold,pseudo_mean_std__strict_hybrid_fold,pseudo_top_entropy__strict_hybrid_fold
0,0.000562,0.000005,0.000390,0.0,0.021824,0.000182,0.015132,0.0,0.023878,0.000199,0.016557,0.0,0.126666,0.001056,0.087830,0.0,0.291046,0.002425,0.201811,0.0,0.536024,0.004467,0.371678,0.0,1.132057,0.536024,0.291046,0.244978,0.537312,38.296250,2.545573,1.102786
1,0.000568,0.000009,0.002001,0.0,0.022032,0.000367,0.077691,0.0,0.024106,0.000402,0.085005,0.0,0.127874,0.002131,0.450923,0.0,0.294571,0.004910,1.038745,0.0,0.530849,0.008847,1.871932,0.0,1.137312,0.530849,0.294571,0.236278,0.526409,37.209297,2.524414,1.108685
2,0.000577,0.000014,0.002418,0.0,0.022402,0.000560,0.093856,0.0,0.024511,0.000613,0.102692,0.0,0.130021,0.003251,0.544744,0.0,0.299517,0.007488,1.254871,0.0,0.522972,0.013074,2.191071,0.0,1.145650,0.522972,0.299517,0.223455,0.509061,35.629297,2.524414,1.117970
3,0.000590,0.000020,0.002709,0.0,0.022884,0.000763,0.105152,0.0,0.025038,0.000835,0.115051,0.0,0.132819,0.004427,0.610307,0.0,0.305186,0.010173,1.402335,0.0,0.513483,0.017116,2.359466,0.0,1.155740,0.513483,0.305186,0.208298,0.488043,33.776250,2.545573,1.129190
4,0.000602,0.000025,0.002936,0.0,0.023378,0.000974,0.113970,0.0,0.025579,0.001066,0.124700,0.0,0.135685,0.005654,0.661488,0.0,0.311770,0.012990,1.519935,0.0,0.502987,0.020958,2.452149,0.0,1.166075,0.502987,0.311770,0.191216,0.466561,31.676250,2.545573,1.140844


## 6. Regime-Specific Mean Models

In [6]:
"Fit global means, regime-specific MAP means, and probability-weighted blends."

BASE_FEATURE_COLS = list(dict.fromkeys([*SAFE_FEATURE_COLS, *TYPEWELL_FEATURE_COLS]))
REGIME_FEATURE_COLS = list(dict.fromkeys([
    *SAFE_FEATURE_COLS,
    *TYPEWELL_FEATURE_COLS,
    "state_known_tvt_slope", "state_known_gr_coverage", "state_known_gr_last", "state_known_gr_iqr",
    "state_known_xy_span", "state_known_z_drop", "state_known_md_span",
]))
REGIME_FEATURE_COLS = [col for col in REGIME_FEATURE_COLS if col in data.columns]

mean_models = ["xgb_square_global", "xgb_pseudohuber_global", "ridge_global"]
ridge_alphas = [80.0, 220.0, 650.0, 1400.0]
blend_schedules = {
    "hard_blend": {"kind": "fixed", "weight": 1.0},
    "s050": {"kind": "fixed", "weight": 0.50},
    "s075": {"kind": "fixed", "weight": 0.75},
    "confidence_s075": {"kind": "confidence", "max_weight": 0.75},
    "confidence_s100": {"kind": "confidence", "max_weight": 1.0},
}


def rmse(y_true, y_pred):
    return float(mean_squared_error(y_true, y_pred) ** 0.5)


def metric_row(actual, pred, baseline):
    return {
        "rmse": rmse(actual, pred),
        "mae": float(mean_absolute_error(actual, pred)),
        "r2": float(r2_score(actual, pred)),
        "last_known_hold_rmse": rmse(actual, baseline),
    }


def fit_global_mean(name, X_train, y_train, X_valid):
    if name == "xgb_square_global":
        model = XGBRegressor(
            objective="reg:squarederror", n_estimators=520, learning_rate=0.036, max_depth=5,
            min_child_weight=90, subsample=0.85, colsample_bytree=0.85, reg_alpha=0.20,
            reg_lambda=8.0, gamma=0.10, tree_method="hist", random_state=SEED, n_jobs=4, verbosity=0,
        )
    elif name == "xgb_pseudohuber_global":
        model = XGBRegressor(
            objective="reg:pseudohubererror", n_estimators=650, learning_rate=0.032, max_depth=4,
            min_child_weight=100, subsample=0.85, colsample_bytree=0.85, reg_alpha=0.15,
            reg_lambda=7.0, gamma=0.05, tree_method="hist", random_state=SEED, n_jobs=4, verbosity=0,
        )
    elif name == "ridge_global":
        model = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), Ridge(alpha=220.0))
    else:
        raise ValueError(name)
    model.fit(X_train, y_train)
    return model.predict(X_train), model.predict(X_valid), model


def fit_weighted_ridge(X_train, y_train, sample_weight, alpha):
    imputer = SimpleImputer(strategy="median")
    scaler = StandardScaler()
    ridge = Ridge(alpha=alpha)
    X_imp = imputer.fit_transform(X_train)
    X_scaled = scaler.fit_transform(X_imp)
    ridge.fit(X_scaled, y_train, sample_weight=sample_weight)
    return {"imputer": imputer, "scaler": scaler, "ridge": ridge}


def predict_weighted_ridge(model, X):
    return model["ridge"].predict(model["scaler"].transform(model["imputer"].transform(X)))


def blend_weights(frame, fold_col, schedule):
    if schedule["kind"] == "fixed":
        return np.full(len(frame), float(schedule["weight"]))
    confidence = frame[f"mix_confidence__{fold_col}"].fillna(frame[f"mix_confidence__{fold_col}"].median()).clip(0.0, 1.0)
    margin = frame[f"mix_prob_margin__{fold_col}"].fillna(frame[f"mix_prob_margin__{fold_col}"].median()).clip(0.0, 1.0)
    entropy = frame[f"mix_entropy__{fold_col}"].fillna(frame[f"mix_entropy__{fold_col}"].median())
    entropy_score = 1.0 / (1.0 + np.exp((entropy - 1.15) / 0.12))
    raw = 0.50 * confidence + 0.35 * margin + 0.15 * entropy_score
    return np.clip(schedule["max_weight"] * np.asarray(raw), 0.0, schedule["max_weight"])


def evaluate_regime_mean(fold_col):
    prob_cols = [f"mix_prob_{formation}__{fold_col}" for formation in FORMATION_COLS]
    X_base = data[BASE_FEATURE_COLS].replace([np.inf, -np.inf], np.nan)
    X_regime = data[REGIME_FEATURE_COLS].replace([np.inf, -np.inf], np.nan)
    y_delta = data["target_delta"].astype(float)
    y_tvt = data["TVT"].astype(float)
    baseline = data["last_known_tvt_feature"].astype(float)
    fold_rows = []
    pred_parts = []
    diag_rows = []
    for fold in sorted(data[fold_col].dropna().unique()):
        valid_mask = data[fold_col].eq(fold)
        train_mask = ~valid_mask
        actual = y_tvt.loc[valid_mask].to_numpy()
        base_tvt = baseline.loc[valid_mask].to_numpy()
        train_probs = data.loc[train_mask, prob_cols].to_numpy(float)
        valid_probs = data.loc[valid_mask, prob_cols].to_numpy(float)
        for global_name in mean_models:
            global_train_delta, global_valid_delta, global_model = fit_global_mean(global_name, X_base.loc[train_mask], y_delta.loc[train_mask], X_base.loc[valid_mask])
            for alpha in ridge_alphas:
                regime_train_preds = []
                regime_valid_preds = []
                for idx, formation in enumerate(FORMATION_COLS):
                    weights = np.clip(train_probs[:, idx], 1e-4, None)
                    weights = weights / np.mean(weights)
                    model = fit_weighted_ridge(X_regime.loc[train_mask], y_delta.loc[train_mask], weights, alpha)
                    regime_train_preds.append(predict_weighted_ridge(model, X_regime.loc[train_mask]))
                    regime_valid_preds.append(predict_weighted_ridge(model, X_regime.loc[valid_mask]))
                regime_train_matrix = np.column_stack(regime_train_preds)
                regime_valid_matrix = np.column_stack(regime_valid_preds)
                blended_train_delta = np.sum(train_probs * regime_train_matrix, axis=1)
                blended_valid_delta = np.sum(valid_probs * regime_valid_matrix, axis=1)
                for schedule_name, schedule in blend_schedules.items():
                    weights = blend_weights(data.loc[valid_mask], fold_col, schedule)
                    pred_delta = (1.0 - weights) * global_valid_delta + weights * blended_valid_delta
                    pred_tvt = base_tvt + pred_delta
                    label = f"{global_name}__regime_ridge_alpha{int(alpha)}__{schedule_name}"
                    metrics = metric_row(actual, pred_tvt, base_tvt)
                    fold_rows.append({
                        "run_id": RUN_ID,
                        "fold_column": fold_col,
                        "fold": int(fold),
                        "label": label,
                        "global_model": global_name,
                        "regime_model": f"regime_ridge_alpha{int(alpha)}",
                        "schedule": schedule_name,
                        "n_train_rows": int(train_mask.sum()),
                        "n_valid_rows": int(valid_mask.sum()),
                        "n_base_features": int(len(BASE_FEATURE_COLS)),
                        "n_regime_features": int(len(REGIME_FEATURE_COLS)),
                        **metrics,
                    })
                    tmp = data.loc[valid_mask, ["wellname", PRIMARY_FOLD, STRESS_FOLD, "eval_progress", "selected_analog_distance", "combined_stress_score", f"mix_entropy__{fold_col}", f"mix_prob_margin__{fold_col}", f"mix_confidence__{fold_col}"]].copy()
                    tmp["fold_column"] = fold_col
                    tmp["fold"] = int(fold)
                    tmp["label"] = label
                    tmp["global_model"] = global_name
                    tmp["regime_model"] = f"regime_ridge_alpha{int(alpha)}"
                    tmp["schedule"] = schedule_name
                    tmp["actual_tvt"] = actual
                    tmp["pred_tvt"] = pred_tvt
                    tmp["baseline_tvt"] = base_tvt
                    tmp["global_pred_tvt"] = base_tvt + global_valid_delta
                    tmp["regime_pred_tvt"] = base_tvt + blended_valid_delta
                    tmp["blend_weight"] = weights
                    tmp["residual"] = actual - pred_tvt
                    pred_parts.append(tmp)
                    diag_rows.append({
                        "fold_column": fold_col,
                        "fold": int(fold),
                        "label": label,
                        "global_model": global_name,
                        "regime_model": f"regime_ridge_alpha{int(alpha)}",
                        "schedule": schedule_name,
                        "mean_blend_weight": float(np.mean(weights)),
                        "max_blend_weight": float(np.max(weights)),
                        "train_global_rmse": rmse(y_delta.loc[train_mask], global_train_delta),
                        "train_regime_rmse": rmse(y_delta.loc[train_mask], blended_train_delta),
                    })
    return pd.DataFrame(fold_rows), pd.concat(pred_parts, ignore_index=True), pd.DataFrame(diag_rows)


start = time.time()
strict_fold_metrics, strict_predictions, strict_diag = evaluate_regime_mean(PRIMARY_FOLD)
stress_fold_metrics, stress_predictions, stress_diag = evaluate_regime_mean(STRESS_FOLD)
print(f"elapsed_s={time.time() - start:.1f}")
strict_fold_metrics.head()

elapsed_s=318.8


,run_id,fold_column,fold,label,global_model,regime_model,schedule,n_train_rows,n_valid_rows,n_base_features,n_regime_features,rmse,mae,r2,last_known_hold_rmse
0,2026-07-01__pseudo_top_state_revision,strict_hybrid_fold,1,xgb_square_global__regime_ridge_alpha80__hard_...,xgb_square_global,regime_ridge_alpha80,hard_blend,61440,31320,41,48,14.270338,10.202061,0.999414,16.418628
1,2026-07-01__pseudo_top_state_revision,strict_hybrid_fold,1,xgb_square_global__regime_ridge_alpha80__s050,xgb_square_global,regime_ridge_alpha80,s050,61440,31320,41,48,13.665231,9.506079,0.999463,16.418628
2,2026-07-01__pseudo_top_state_revision,strict_hybrid_fold,1,xgb_square_global__regime_ridge_alpha80__s075,xgb_square_global,regime_ridge_alpha80,s075,61440,31320,41,48,13.860935,9.763626,0.999447,16.418628
3,2026-07-01__pseudo_top_state_revision,strict_hybrid_fold,1,xgb_square_global__regime_ridge_alpha80__confi...,xgb_square_global,regime_ridge_alpha80,confidence_s075,61440,31320,41,48,13.616986,9.456016,0.999467,16.418628
4,2026-07-01__pseudo_top_state_revision,strict_hybrid_fold,1,xgb_square_global__regime_ridge_alpha80__confi...,xgb_square_global,regime_ridge_alpha80,confidence_s100,61440,31320,41,48,13.589707,9.468803,0.999469,16.418628


## 7. Results And Diagnostics

In [7]:
"Summarize strict-CV, stress-CV, and regime-mean behavior."

strict_summary = (
    strict_fold_metrics.groupby(["label", "global_model", "regime_model", "schedule"])
    .agg(
        rmse_mean=("rmse", "mean"),
        rmse_std=("rmse", "std"),
        mae_mean=("mae", "mean"),
        r2_mean=("r2", "mean"),
        last_known_hold_rmse_mean=("last_known_hold_rmse", "mean"),
        n_valid_rows=("n_valid_rows", "sum"),
        n_base_features=("n_base_features", "max"),
        n_regime_features=("n_regime_features", "max"),
    )
    .reset_index()
)
stress_summary = (
    stress_fold_metrics.groupby(["label", "global_model", "regime_model", "schedule"])
    .agg(
        stress_rmse_mean=("rmse", "mean"),
        stress_rmse_std=("rmse", "std"),
        stress_mae_mean=("mae", "mean"),
        stress_last_known_hold_rmse_mean=("last_known_hold_rmse", "mean"),
        stress_n_valid_rows=("n_valid_rows", "sum"),
    )
    .reset_index()
)
summary = strict_summary.merge(stress_summary, on=["label", "global_model", "regime_model", "schedule"], how="left")
summary["rmse_delta_vs_hold"] = summary["rmse_mean"] - summary["last_known_hold_rmse_mean"]
summary["rmse_delta_vs_mixture_prior"] = summary["rmse_mean"] - 13.443736843906953
summary["rmse_delta_vs_safe_typewell_reference"] = summary["rmse_mean"] - 13.450635286744287
summary["rmse_delta_vs_public_best"] = summary["rmse_mean"] - CONFIG["current_public_best_rmse"]
summary["stress_minus_strict_rmse"] = summary["stress_rmse_mean"] - summary["rmse_mean"]
summary = summary.sort_values("rmse_mean").reset_index(drop=True)
best_row = summary.iloc[0].copy()
best_label = str(best_row["label"])

predictions = pd.concat([strict_predictions, stress_predictions], ignore_index=True)
predictions["progress_zone"] = pd.cut(predictions["eval_progress"], bins=[-0.001, 0.25, 0.50, 0.75, 1.001], labels=["z0", "z1", "z2", "z3"])
predictions["analog_distance_bin"] = pd.qcut(predictions["selected_analog_distance"], q=5, labels=["closest", "near", "middle", "far", "farthest"], duplicates="drop")

zone_summary = (
    predictions.groupby(["fold_column", "label", "progress_zone"], observed=True)
    .apply(lambda group: pd.Series({
        "n_rows": len(group),
        "rmse": rmse(group["actual_tvt"], group["pred_tvt"]),
        "global_rmse": rmse(group["actual_tvt"], group["global_pred_tvt"]),
        "regime_rmse": rmse(group["actual_tvt"], group["regime_pred_tvt"]),
        "baseline_rmse": rmse(group["actual_tvt"], group["baseline_tvt"]),
        "mean_blend_weight": float(group["blend_weight"].mean()),
        "mean_mix_confidence": float(group.filter(like="mix_confidence").iloc[:, 0].mean()),
    }), include_groups=False)
    .reset_index()
)
zone_summary["rmse_delta_vs_global"] = zone_summary["rmse"] - zone_summary["global_rmse"]

analog_bin_summary = (
    predictions.groupby(["fold_column", "label", "analog_distance_bin"], observed=True)
    .apply(lambda group: pd.Series({
        "n_rows": len(group),
        "n_wells": group["wellname"].nunique(),
        "rmse": rmse(group["actual_tvt"], group["pred_tvt"]),
        "global_rmse": rmse(group["actual_tvt"], group["global_pred_tvt"]),
        "regime_rmse": rmse(group["actual_tvt"], group["regime_pred_tvt"]),
        "baseline_rmse": rmse(group["actual_tvt"], group["baseline_tvt"]),
        "mean_blend_weight": float(group["blend_weight"].mean()),
    }), include_groups=False)
    .reset_index()
)
analog_bin_summary["rmse_delta_vs_global"] = analog_bin_summary["rmse"] - analog_bin_summary["global_rmse"]

well_diagnostics = (
    predictions.loc[(predictions["fold_column"].eq(PRIMARY_FOLD)) & (predictions["label"].eq(best_label))]
    .groupby("wellname")
    .agg(
        n_rows=("actual_tvt", "size"),
        strict_fold=(PRIMARY_FOLD, "first"),
        analog_fold=(STRESS_FOLD, "first"),
        mean_selected_analog_distance=("selected_analog_distance", "mean"),
        mean_blend_weight=("blend_weight", "mean"),
        mean_residual=("residual", "mean"),
        mean_abs_residual=("residual", lambda series: float(np.abs(series).mean())),
        rmse=("residual", lambda series: float(np.sqrt(np.mean(np.square(series))))),
    )
    .reset_index()
    .sort_values("rmse", ascending=False)
)

regime_diagnostics = pd.concat([strict_diag, stress_diag], ignore_index=True)
regime_summary = (
    regime_diagnostics.groupby(["fold_column", "label", "global_model", "regime_model", "schedule"])
    .agg(
        mean_blend_weight=("mean_blend_weight", "mean"),
        max_blend_weight=("max_blend_weight", "max"),
        train_global_rmse=("train_global_rmse", "mean"),
        train_regime_rmse=("train_regime_rmse", "mean"),
    )
    .reset_index()
)

print(summary.head(18).to_string(index=False))

                                                         label           global_model           regime_model        schedule  rmse_mean  rmse_std  mae_mean  r2_mean  last_known_hold_rmse_mean  n_valid_rows  n_base_features  n_regime_features  stress_rmse_mean  stress_rmse_std  stress_mae_mean  stress_last_known_hold_rmse_mean  stress_n_valid_rows  rmse_delta_vs_hold  rmse_delta_vs_mixture_prior  rmse_delta_vs_safe_typewell_reference  rmse_delta_vs_public_best  stress_minus_strict_rmse
                 xgb_square_global__regime_ridge_alpha80__s050      xgb_square_global   regime_ridge_alpha80            s050  13.119439  0.524776  9.368172 0.999546                  15.571277         92760               41                 48         13.388295         0.698695         9.718642                         15.581239                92760           -2.451838                    -0.324297                              -0.331196                  -1.352561                  0.268856
      xgb_square_glo

## 8. Artifact Export

In [8]:
"Export artifacts, report, and model-comparison row."

best_strict_rmse = float(best_row["rmse_mean"])
best_stress_rmse = float(best_row["stress_rmse_mean"])
best_global = summary.loc[summary["schedule"].eq("s050")].sort_values("rmse_mean").iloc[0]
best_confidence = summary.loc[summary["schedule"].str.contains("confidence", regex=False)].sort_values("rmse_mean").iloc[0]

if best_strict_rmse < CONFIG["baseline_regime_mean_rmse"] - CONFIG["promotion_margin"] and best_stress_rmse <= CONFIG["baseline_regime_mean_stress_rmse"]:
    recommendation = "package_pseudo_top_state_revision"
elif best_stress_rmse < CONFIG["baseline_regime_mean_stress_rmse"] and best_strict_rmse <= CONFIG["baseline_regime_mean_rmse"] + 0.05:
    recommendation = "stress_diagnostic_iterate"
elif best_strict_rmse < CONFIG["current_public_best_rmse"]:
    recommendation = "diagnostic_iterate_before_package"
else:
    recommendation = "diagnostic_only"

findings = pd.DataFrame([
    {"finding": "best_overall", "detail": best_label, "value": best_strict_rmse, "interpretation": f"Best strict RMSE with analog stress RMSE {best_stress_rmse:.4f}."},
    {"finding": "best_fixed_blend", "detail": str(best_global["label"]), "value": float(best_global["rmse_mean"]), "interpretation": "Best fixed fallback/blend candidate."},
    {"finding": "best_confidence_blend", "detail": str(best_confidence["label"]), "value": float(best_confidence["rmse_mean"]), "interpretation": "Best confidence-gated regime mean candidate."},
    {"finding": "decision", "detail": recommendation, "value": best_strict_rmse, "interpretation": "Promotion rule checks mixture-prior reference, stress behavior, and current public context."},
])

paths = {
    "summary": TABLE_DIR / "model__pseudo_top_state_revision_summary__2026-07-01.csv",
    "fold_metrics": TABLE_DIR / "model__pseudo_top_state_revision_fold_metrics__2026-07-01.csv",
    "regime_diagnostics": TABLE_DIR / "model__pseudo_top_state_revision_regime_diagnostics__2026-07-01.csv",
    "revision_quality": TABLE_DIR / "model__pseudo_top_state_revision_revision_quality__2026-07-01.csv",
    "zone_summary": TABLE_DIR / "model__pseudo_top_state_revision_zone_summary__2026-07-01.csv",
    "analog_bin_summary": TABLE_DIR / "model__pseudo_top_state_revision_analog_bin_summary__2026-07-01.csv",
    "well_diagnostics": TABLE_DIR / "model__pseudo_top_state_revision_well_diagnostics__2026-07-01.csv",
    "findings": TABLE_DIR / "model__pseudo_top_state_revision_findings__2026-07-01.csv",
}
exports = {
    "summary": summary,
    "fold_metrics": pd.concat([strict_fold_metrics, stress_fold_metrics], ignore_index=True),
    "regime_diagnostics": regime_summary,
    "revision_quality": revision_quality,
    "zone_summary": zone_summary,
    "analog_bin_summary": analog_bin_summary,
    "well_diagnostics": well_diagnostics,
    "findings": findings,
}
for key, frame in exports.items():
    frame.to_csv(paths[key], index=False)
    shutil.copy2(paths[key], RUN_DIR / paths[key].name)

comparison_path = TABLE_DIR / "model_comparison.csv"
comparison = pd.read_csv(comparison_path) if comparison_path.exists() else pd.DataFrame()
new_row = pd.DataFrame([{
    "run_id": RUN_ID,
    "model_family": "bayesian",
    "model_name": "pseudo_top_state_revision",
    "best_candidate": best_label,
    "local_cv_rmse": best_strict_rmse,
    "local_cv_mae": float(best_row["mae_mean"]),
    "public_lb_rmse": np.nan,
    "status": recommendation,
    "notes": f"stress_rmse={best_stress_rmse:.4f}; best_confidence={float(best_confidence['rmse_mean']):.4f}",
    "updated_at": RUN_DATE,
}])
if not comparison.empty and "run_id" in comparison.columns:
    comparison = comparison.loc[comparison["run_id"] != RUN_ID].copy()
comparison = pd.concat([comparison, new_row], ignore_index=True, sort=False)
comparison.to_csv(comparison_path, index=False)
shutil.copy2(comparison_path, RUN_DIR / comparison_path.name)

report_path = REPORT_DIR / "model__pseudo_top_state_revision__2026-07-01.md"
report = f"""# Pseudo-Top State Revision

## Result

Best overall candidate: `{best_label}` with strict-CV RMSE {best_strict_rmse:.4f}, MAE {float(best_row["mae_mean"]):.4f}, and analog-distance stress RMSE {best_stress_rmse:.4f}.

Best confidence-gated regime mean candidate: `{best_confidence["label"]}` with RMSE {float(best_confidence["rmse_mean"]):.4f}.

## Model Structure

Pseudo-top predictions are first corrected by conservative fold-safe residual models, then converted into soft formation regimes for the direct regime-mean model.

## Leakage Notes

Direct formation columns are used only as fold-local labels for pseudo-top proxy training. Regime probabilities and regime means use held-out pseudo-top predictions and hidden-test-safe features.

## Decision

Recommendation: `{recommendation}`.

## Next Move

If this does not beat the residual mixture prior, prioritize validation/public-transfer calibration or a heavier hierarchical Bayesian mean with shared regime priors.
"""
report_path.write_text(report, encoding="utf-8")
shutil.copy2(report_path, RUN_DIR / report_path.name)

print(findings.to_string(index=False))
print(report_path)

              finding                                                   detail     value                                                                              interpretation
         best_overall            xgb_square_global__regime_ridge_alpha80__s050 13.119439                                           Best strict RMSE with analog stress RMSE 13.3883.
     best_fixed_blend            xgb_square_global__regime_ridge_alpha80__s050 13.119439                                                        Best fixed fallback/blend candidate.
best_confidence_blend xgb_square_global__regime_ridge_alpha80__confidence_s100 13.127954                                                Best confidence-gated regime mean candidate.
             decision                        package_pseudo_top_state_revision 13.119439 Promotion rule checks mixture-prior reference, stress behavior, and current public context.


C:\Users\sardo\OneDrive\Desktop\Classes\Projects\Wellbore-tvt-Modeling\artifacts\reports\model__pseudo_top_state_revision__2026-07-01.md


## 9. Findings

In [9]:
"Show headline result tables."

display(summary.head(15))
display(regime_summary.loc[regime_summary["label"].eq(best_label)].head())
display(findings)

,label,global_model,regime_model,schedule,rmse_mean,rmse_std,mae_mean,r2_mean,last_known_hold_rmse_mean,n_valid_rows,n_base_features,n_regime_features,stress_rmse_mean,stress_rmse_std,stress_mae_mean,stress_last_known_hold_rmse_mean,stress_n_valid_rows,rmse_delta_vs_hold,rmse_delta_vs_mixture_prior,rmse_delta_vs_safe_typewell_reference,rmse_delta_vs_public_best,stress_minus_strict_rmse
0,xgb_square_global__regime_ridge_alpha80__s050,xgb_square_global,regime_ridge_alpha80,s050,13.119439,0.524776,9.368172,0.999546,15.571277,92760,41,48,13.388295,0.698695,9.718642,15.581239,92760,-2.451838,-0.324297,-0.331196,-1.352561,0.268856
1,xgb_square_global__regime_ridge_alpha80__confi...,xgb_square_global,regime_ridge_alpha80,confidence_s100,13.127954,0.426766,9.382740,0.999546,15.571277,92760,41,48,13.401303,0.674996,9.746039,15.581239,92760,-2.443323,-0.315782,-0.322681,-1.344046,0.273349
2,xgb_square_global__regime_ridge_alpha220__s050,xgb_square_global,regime_ridge_alpha220,s050,13.130934,0.535817,9.373587,0.999545,15.571277,92760,41,48,13.370968,0.647037,9.704621,15.581239,92760,-2.440343,-0.312803,-0.319701,-1.341066,0.240034
3,xgb_square_global__regime_ridge_alpha220__conf...,xgb_square_global,regime_ridge_alpha220,confidence_s100,13.138122,0.439648,9.385909,0.999545,15.571277,92760,41,48,13.388482,0.630994,9.736326,15.581239,92760,-2.433155,-0.305614,-0.312513,-1.333878,0.250360
4,xgb_square_global__regime_ridge_alpha80__confi...,xgb_square_global,regime_ridge_alpha80,confidence_s075,13.138127,0.433495,9.355545,0.999545,15.571277,92760,41,48,13.406853,0.616006,9.738856,15.581239,92760,-2.433150,-0.305610,-0.312509,-1.333873,0.268727
5,xgb_square_global__regime_ridge_alpha220__conf...,xgb_square_global,regime_ridge_alpha220,confidence_s075,13.148639,0.442158,9.360174,0.999544,15.571277,92760,41,48,13.405777,0.592915,9.735052,15.581239,92760,-2.422638,-0.295098,-0.301996,-1.323361,0.257138
6,xgb_square_global__regime_ridge_alpha650__s050,xgb_square_global,regime_ridge_alpha650,s050,13.170036,0.557742,9.394860,0.999542,15.571277,92760,41,48,13.364481,0.569411,9.691892,15.581239,92760,-2.401241,-0.273701,-0.280600,-1.301964,0.194445
7,xgb_square_global__regime_ridge_alpha650__conf...,xgb_square_global,regime_ridge_alpha650,confidence_s100,13.172899,0.465641,9.401423,0.999542,15.571277,92760,41,48,13.391483,0.572475,9.729232,15.581239,92760,-2.398378,-0.270838,-0.277736,-1.299101,0.218584
8,xgb_square_global__regime_ridge_alpha650__conf...,xgb_square_global,regime_ridge_alpha650,confidence_s075,13.180446,0.459635,9.376446,0.999542,15.571277,92760,41,48,13.422495,0.562289,9.737416,15.581239,92760,-2.390831,-0.263291,-0.270189,-1.291554,0.242049
9,xgb_pseudohuber_global__regime_ridge_alpha80__...,xgb_pseudohuber_global,regime_ridge_alpha80,confidence_s100,13.222702,0.708207,9.431433,0.999537,15.571277,92760,41,48,13.503666,0.714621,9.728766,15.581239,92760,-2.348575,-0.221035,-0.227933,-1.249298,0.280963


,fold_column,label,global_model,regime_model,schedule,mean_blend_weight,max_blend_weight,train_global_rmse,train_regime_rmse
58,analog_distance_fold,xgb_square_global__regime_ridge_alpha80__s050,xgb_square_global,regime_ridge_alpha80,s050,0.5,0.5,5.321482,13.187567
118,strict_hybrid_fold,xgb_square_global__regime_ridge_alpha80__s050,xgb_square_global,regime_ridge_alpha80,s050,0.5,0.5,5.353068,13.194415


,finding,detail,value,interpretation
0,best_overall,xgb_square_global__regime_ridge_alpha80__s050,13.119439,Best strict RMSE with analog stress RMSE 13.3883.
1,best_fixed_blend,xgb_square_global__regime_ridge_alpha80__s050,13.119439,Best fixed fallback/blend candidate.
2,best_confidence_blend,xgb_square_global__regime_ridge_alpha80__confi...,13.127954,Best confidence-gated regime mean candidate.
3,decision,package_pseudo_top_state_revision,13.119439,"Promotion rule checks mixture-prior reference,..."
